<a href="https://colab.research.google.com/github/sohampatilakl/Bank-Customer-Churn-Prediction/blob/main/Bank_Customer_Churn_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# 1. Load the dataset
url = "https://raw.githubusercontent.com/selva86/datasets/master/Churn_Modelling.csv"
df = pd.read_csv(url)

# 2. Edit Customer IDs to start sequentially from 151001
start_id = 151001
df['CustomerId'] = range(start_id, start_id + len(df))

# 3. Add Complaints columns
np.random.seed(42)
df['NumOfComplaints'] = np.random.randint(0, 4, size=len(df))  # Used as 'Open Complaints'
df['ClosedComplaints'] = np.random.randint(0, 15, size=len(df)) # Past resolved complaints

print("Data loaded successfully. Here is a sample of the new Customer IDs you can test:")
print(df['CustomerId'].head(5).tolist())

Data loaded successfully. Here is a sample of the new Customer IDs you can test:
[151001, 151002, 151003, 151004, 151005]


In [ ]:
# 1. Save original dataframe for the interface lookup, create a copy for modeling
df_model = df.copy()

# 2. Drop identifiers from the modeling data
df_model = df_model.drop(columns=['RowNumber', 'CustomerId', 'Surname'])

# 3. One-Hot Encode categorical variables
df_model = pd.get_dummies(df_model, columns=['Geography', 'Gender'], drop_first=True)

# 4. Separate Features (X) and Target (y)
X = df_model.drop('Exited', axis=1)
y = df_model['Exited']

# 5. Standardize numerical features
scaler = StandardScaler()
numerical_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary', 'NumOfComplaints']
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])

# 6. Train the Random Forest Model
# We train on the whole dataset here since our goal is building the interface tool
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42)
rf_model.fit(X, y)

print("Model trained successfully! Ready for customer lookups.")

Model trained successfully! Ready for customer lookups.


In [ ]:
def check_customer_churn_risk(customer_id):
    # 1. Check if customer exists in our database
    if customer_id not in df['CustomerId'].values:
        return "Error: Customer ID not found in the database."

    # 2. Get the customer's original data
    customer_info = df[df['CustomerId'] == customer_id].iloc[0]

    # 3. Get the pre-processed features for this specific customer to feed the model
    customer_index = df[df['CustomerId'] == customer_id].index[0]
    customer_features = X.iloc[[customer_index]]

    # 4. Predict the probability of churn
    churn_probability = rf_model.predict_proba(customer_features)[0][1] * 100

    # 5. Display the Full Dashboard
    print("=" * 50)
    print(" 🏦 CUSTOMER RETENTION DASHBOARD")
    print("=" * 50)
    print("--- CUSTOMER PROFILE ---")

    # Dynamically print all general customer information
    for column_name, value in customer_info.items():
        # Skip the internal complaint/exit columns here so we can highlight them at the end
        if column_name not in ['NumOfComplaints', 'ClosedComplaints', 'Exited']:
            print(f"{column_name:<18}: {value}")

    print("-" * 50)
    print("--- COMPLAINTS & CHURN RISK ---")
    print(f"Open Complaints   : {customer_info['NumOfComplaints']}")
    print(f"Closed Complaints : {customer_info['ClosedComplaints']}")
    print(f"Churn Risk        : {churn_probability:.1f}%")
    print("-" * 50)

    # 6. Apply the Business Logic (30% Threshold)
    if churn_probability > 30.0:
        print("💡 ACTION REQUIRED: HIGH RISK")
        print("Generate targeted retention offer immediately.")
        print("Tailor offer based on customer profile.")
    else:
        print("✅ ACTION REQUIRED: LOW RISK")
        print("No special retention offers needed at this time.")
    print("=" * 50)

# ==========================================
# Run the Interface
# ==========================================
# Try these new examples: 151001, 151002, 151003
input_id = int(input("Enter Customer ID to check: "))
check_customer_churn_risk(input_id)

Enter Customer ID to check: 151009
 🏦 CUSTOMER RETENTION DASHBOARD
--- CUSTOMER PROFILE ---
RowNumber         : 9
CustomerId        : 151009
Surname           : He
CreditScore       : 501
Geography         : France
Gender            : Male
Age               : 44
Tenure            : 4
Balance           : 142051.07
NumOfProducts     : 2
HasCrCard         : 0
IsActiveMember    : 1
EstimatedSalary   : 74940.5
--------------------------------------------------
--- COMPLAINTS & CHURN RISK ---
Open Complaints   : 2
Closed Complaints : 10
Churn Risk        : 38.4%
--------------------------------------------------
💡 ACTION REQUIRED: HIGH RISK
Generate targeted retention offer immediately.
Tailor offer based on customer profile.
